<a href="https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [9]:
%pip -q install duckdb huggingface_hub


In [10]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [11]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [12]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [15]:
features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last45,
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END) AS pos_last45
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 50
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features_90):,} content items with enough history (90-day window)')
features_90.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

142,816 content items with enough history (90-day window)


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_last45
0,client_e547b89c05043229,content_1557a3abbc832229,316.0,184.0,2.0,15.632317
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,355.0,394.0,0.0,24.035644
2,client_e547b89c05043229,content_48537762b74f5b34,288.0,463.0,0.0,27.790221
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,254.0,304.0,1.0,23.670358
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,763.0,1075.0,1.0,32.625971


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [17]:
features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last45,
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END) AS pos_last45,
            STDDEV(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END) AS pos_volatility_last45
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 50
    )
    SELECT * FROM windowed
""").df()
print(f'{len(features_90):,} content items with enough history (90-day window)')
features_90.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

142,816 content items with enough history (90-day window)


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_last45,pos_volatility_last45
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,27.0,87.0,1.0,5.129630,1.979928
1,client_3ffa76342f366962,content_f33fd2cd160185e7,64.0,107.0,1.0,6.114943,2.535405
2,client_e547b89c05043229,content_2e296120acb03e93,4097.0,3456.0,0.0,39.885483,6.964421
3,client_e547b89c05043229,content_516b7c0e8eec0cef,633.0,2232.0,0.0,53.022749,15.726513
4,client_e547b89c05043229,content_38b6c1a9aa29f801,8284.0,10119.0,2.0,41.777873,6.434030


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [18]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.537     0.338     0.415      9389
           1      0.684     0.831     0.750     16162

    accuracy                          0.650     25551
   macro avg      0.610     0.584     0.583     25551
weighted avg      0.630     0.650     0.627     25551



In [19]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))

X_tr2, X_te2 = X.iloc[train_idx], X.iloc[test_idx]
y_tr2, y_te2 = y.iloc[train_idx], y.iloc[test_idx]

model2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)

print(f'base rate (always predict majority): {max(y_te2.mean(), 1 - y_te2.mean()):.3f}')
print(classification_report(y_te2, model2.predict(X_te2), digits=3))

base rate (always predict majority): 0.677
              precision    recall  f1-score   support

           0      0.413     0.463     0.436     16706
           1      0.728     0.686     0.707     34996

    accuracy                          0.614     51702
   macro avg      0.571     0.574     0.571     51702
weighted avg      0.626     0.614     0.619     51702



## Findings: Random split vs GroupShuffleSplit

- Random split: accuracy 0.650 vs base rate 0.633 (model barely beats baseline)
- Client-grouped split (GroupShuffleSplit): accuracy 0.614 vs base rate 0.677
  (model actually performs WORSE than just guessing the majority class)

This is a meaningful gap. It suggests the random-split model was partly learning
client-specific patterns rather than a generalizable rule — when tested on entirely
unseen clients, it doesn't hold up. This matches the warning in the notebook:
"does it generalize across clients?" is the real test, and here the answer is no,
not yet. A next step would be looking at which features are driving predictions
and whether client-level normalization (e.g. relative rather than absolute impression
counts) might help the model generalize better.

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
